# 05 — API Testing: Smoke Tests, Latency Benchmarks, and Stress Testing

**Covers:**
- `notes/02-advanced-deep-learning/ch11-deployment/` — serving the detection model via REST API
- Integration testing strategy: from health check → single inference → batch → concurrency

**Goal:** Validate that the Flask API in `src/api.py` is behaving correctly end-to-end: correct response schema, reasonable latency, stable under concurrent load.

**Prerequisites:**
- The Flask API server must be running before executing cells 3–12
- Start it with: `python src/api.py` (or via `config.yaml` launch script)
- `requests`, `httpx`, `asyncio` installed (standard Python stdlib for asyncio)

**`QUICK_MODE`:** Uses a mock server that answers `/health` and `/detect` locally without a real model, so you can explore the test structure without starting the actual service.

**`SERVICE_URL`:** Change this to point at a remote deployment (Azure Container Instance, Jetson Nano IP, etc.).

In [ ]:
# ── Imports & service configuration ──────────────────────────────────────────
import asyncio
import io
import json
import random
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print('requests not installed. Run: pip install requests')

try:
    import httpx
    HAS_HTTPX = True
except ImportError:
    HAS_HTTPX = False
    print('httpx not installed. Run: pip install httpx')

from PIL import Image

# QUICK_MODE=True: uses a local mock server (no external process needed)
# QUICK_MODE=False: tests the real Flask API running at SERVICE_URL
QUICK_MODE  = False
SERVICE_URL = 'http://localhost:5000'  # change to remote host:port if needed
TIMEOUT_S   = 30    # per-request timeout

print(f'QUICK_MODE={QUICK_MODE}')
print(f'SERVICE_URL={SERVICE_URL}')
print(f'requests={HAS_REQUESTS}, httpx={HAS_HTTPX}')

## Setup — Mock Server for QUICK_MODE

The mock server is a lightweight in-process stub that returns correctly-structured responses without actually loading the model. It lets you validate:
1. The test harness is wired up correctly
2. Response schema assertions work as expected
3. Latency measurement code is correct

In `QUICK_MODE=False`, skip this cell — we'll hit the real service instead.

In [ ]:
# ── Mock server for QUICK_MODE ────────────────────────────────────────────────
# The mock intercepts `requests.get` / `requests.post` calls and returns
# pre-canned responses that match the real API schema in src/api.py.
# This avoids needing to start the Flask server for quick notebook testing.

import unittest.mock as mock

MOCK_HEALTH_RESPONSE = {
    'status': 'healthy',
    'version': '1.0.0-mock',
    'model_loaded': True,
    'device': 'cpu',
}

def _make_mock_detections(n=3):
    """Generate a realistic /detect response payload."""
    cat_names = ['product', 'shelf', 'price_tag', 'empty_slot', 'label']
    return {
        'detections': [
            {
                'bbox':       [random.randint(10, 200), random.randint(10, 200),
                               random.randint(30, 150), random.randint(30, 150)],
                'label':      random.choice(cat_names),
                'confidence': round(random.uniform(0.55, 0.99), 3),
                'category_id': random.randint(1, 5),
            }
            for _ in range(n)
        ],
        'inference_ms': round(random.uniform(18, 45), 2),
        'image_id':     f'test_{random.randint(1000, 9999)}',
    }

class MockResponse:
    """Minimal requests.Response mock."""
    def __init__(self, data, status_code=200):
        self._data = data
        self.status_code = status_code
    def json(self):
        return self._data
    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f'{self.status_code}')

def mock_get(url, **kwargs):
    if url.endswith('/health'):
        return MockResponse(MOCK_HEALTH_RESPONSE)
    return MockResponse({'error': 'not found'}, 404)

def mock_post(url, **kwargs):
    # Simulate variable latency
    time.sleep(random.uniform(0.010, 0.045))
    if url.endswith('/detect'):
        return MockResponse(_make_mock_detections())
    return MockResponse({'error': 'not found'}, 404)

if QUICK_MODE and HAS_REQUESTS:
    # Patch requests globally so all subsequent cells use the mock
    requests.get  = mock_get
    requests.post = mock_post
    print('[QUICK_MODE] Mock server active — requests intercepted locally')
elif not QUICK_MODE:
    print(f'[Real mode] Will connect to {SERVICE_URL}')
    print('Ensure Flask server is running: python src/api.py')

## 1. Health Check — Is the Server Up?

The health endpoint (`GET /health`) is the first thing any monitoring system checks. It must:
- Return HTTP 200
- Respond in < 100ms (no model inference involved)
- Include version info so we can verify the right model is deployed

In production, this endpoint is polled by Kubernetes liveness probes or an Azure Load Balancer every 10 seconds. If it returns anything other than 200, the service is taken out of rotation.

In [ ]:
# ── GET /health — assert 200, print version ───────────────────────────────────
if HAS_REQUESTS:
    t0 = time.perf_counter()
    response = requests.get(f'{SERVICE_URL}/health', timeout=TIMEOUT_S)
    health_latency_ms = (time.perf_counter() - t0) * 1000

    # Hard assertion: 200 is the only acceptable status
    assert response.status_code == 200, (
        f'Expected HTTP 200, got {response.status_code}. '
        'Check that the API server is running.'
    )

    health_data = response.json()
    print('✓ Health check PASSED')
    print(f'  HTTP status:      {response.status_code}')
    print(f'  Latency:          {health_latency_ms:.1f}ms')
    print(f'  Server version:   {health_data.get("version", "N/A")}')
    print(f'  Model loaded:     {health_data.get("model_loaded", "N/A")}')
    print(f'  Device:           {health_data.get("device", "N/A")}')

    # Warn if health check itself is slow — might indicate resource contention
    if health_latency_ms > 100:
        print(f'  ⚠  Health check latency {health_latency_ms:.0f}ms > 100ms')
else:
    print('[Skipped — install requests to run this cell]')

## 2. Single Inference — Smoke Test

**Smoke test:** Send one image, verify the response structure is correct. This catches:
- Wrong JSON schema (missing fields, wrong types)
- Model not loaded (inference error instead of detections)
- MIME type or encoding issues in the request

The API contract (from `src/api.py`):
- Request: `POST /detect` with `Content-Type: image/jpeg` body (or multipart form with `file` key)
- Response: `{detections: [{bbox, label, confidence, category_id}], inference_ms, image_id}`

We validate the schema programmatically so this can be wired into a CI pipeline.

In [ ]:
# ── POST /detect — smoke test + bbox overlay visualization ────────────────────
def make_test_image_bytes(width=640, height=480, seed=42):
    """Create a synthetic JPEG image (random noise) encoded as bytes."""
    rng = np.random.RandomState(seed)
    img_array = (rng.rand(height, width, 3) * 255).astype(np.uint8)
    # Draw a few product-like rectangles so the image has structure
    for _ in range(5):
        x1, y1 = rng.randint(0, width//2), rng.randint(0, height//2)
        x2, y2 = x1 + rng.randint(50, 150), y1 + rng.randint(50, 150)
        img_array[y1:y2, x1:x2] = rng.randint(50, 200, 3, dtype=np.uint8)
    pil_img = Image.fromarray(img_array)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=85)
    return buf.getvalue(), img_array

def validate_detection_response(response_json):
    """Assert the /detect response matches the expected schema."""
    assert 'detections' in response_json, 'Missing detections field'
    assert 'inference_ms' in response_json, 'Missing inference_ms field'
    assert isinstance(response_json['detections'], list), 'detections must be a list'
    for det in response_json['detections']:
        assert 'bbox' in det,       'Detection missing bbox'
        assert 'label' in det,      'Detection missing label'
        assert 'confidence' in det, 'Detection missing confidence'
        assert len(det['bbox']) == 4, 'bbox must have 4 values [x, y, w, h]'
        assert 0.0 <= det['confidence'] <= 1.0, 'confidence out of [0, 1] range'
    return True

if HAS_REQUESTS:
    img_bytes, img_array = make_test_image_bytes(seed=42)

    t0 = time.perf_counter()
    response = requests.post(
        f'{SERVICE_URL}/detect',
        data=img_bytes,
        headers={'Content-Type': 'image/jpeg'},
        timeout=TIMEOUT_S,
    )
    round_trip_ms = (time.perf_counter() - t0) * 1000

    assert response.status_code == 200, f'Expected 200, got {response.status_code}'
    result = response.json()
    validate_detection_response(result)

    print(f'✓ Smoke test PASSED')
    print(f'  Round-trip latency:  {round_trip_ms:.1f}ms')
    print(f'  Model inference:     {result.get("inference_ms", "N/A")}ms')
    print(f'  Detections returned: {len(result["detections"])}')

    # Visualize bounding box overlay
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img_array)
    colors = plt.cm.Set1.colors
    for i, det in enumerate(result['detections']):
        bx, by, bw, bh = det['bbox']
        color = colors[i % len(colors)]
        rect = mpatches.Rectangle((bx, by), bw, bh,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(bx + 2, max(by - 6, 0),
                f'{det["label"]} {det["confidence"]:.2f}',
                color='white', fontsize=8, weight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.7))
    ax.set_title(f'Detection results — {len(result["detections"])} objects found', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Batch Inference — Throughput Measurement

**Throughput vs latency:** A single request might be fast (20ms), but what happens when the retail camera is streaming 50 images per minute? We simulate this by sending 50 requests sequentially and measuring per-image latency.

**Bottleneck identification:**
- If latency is consistent → throughput is bounded by model inference time
- If latency *increases* over the 50-image sequence → server-side memory leak or GC pressure
- If some requests are 10× slower → contention on a shared GPU resource

In [ ]:
# ── Sequential batch: 50 images through /detect ───────────────────────────────
N_IMAGES = 20 if QUICK_MODE else 50

if HAS_REQUESTS:
    per_image_latencies = []
    detection_counts    = []

    for i in range(N_IMAGES):
        img_bytes, _ = make_test_image_bytes(seed=i)
        t0 = time.perf_counter()
        resp = requests.post(
            f'{SERVICE_URL}/detect',
            data=img_bytes,
            headers={'Content-Type': 'image/jpeg'},
            timeout=TIMEOUT_S,
        )
        latency_ms = (time.perf_counter() - t0) * 1000

        if resp.status_code == 200:
            per_image_latencies.append(latency_ms)
            detection_counts.append(len(resp.json().get('detections', [])))
        else:
            print(f'  Image {i}: HTTP {resp.status_code}')

    lat = np.array(per_image_latencies)
    print(f'Batch throughput test ({N_IMAGES} sequential requests)')
    print(f'  Mean latency:   {lat.mean():.1f}ms')
    print(f'  Median latency: {np.median(lat):.1f}ms')
    print(f'  P95 latency:    {np.percentile(lat, 95):.1f}ms')
    print(f'  Min / Max:      {lat.min():.1f}ms / {lat.max():.1f}ms')
    print(f'  Throughput:     {1000/lat.mean():.1f} images/sec')
    print(f'  Avg detections: {np.mean(detection_counts):.1f} per image')

    # Plot: latency over time (reveals drift, spikes, GC pauses)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(range(len(lat)), lat, alpha=0.7, color='steelblue')
    axes[0].axhline(np.median(lat), color='red', linestyle='--', label=f'Median {np.median(lat):.0f}ms')
    axes[0].axhline(50, color='orange', linestyle=':', label='50ms target')
    axes[0].set_xlabel('Request #'); axes[0].set_ylabel('Latency (ms)')
    axes[0].set_title('Per-Request Latency Over Time')
    axes[0].legend(fontsize=8)
    axes[1].hist(lat, bins=15, color='coral', edgecolor='white')
    axes[1].set_xlabel('Latency (ms)'); axes[1].set_ylabel('Count')
    axes[1].set_title('Latency Distribution')
    plt.tight_layout()
    plt.show()

## 4. Async Stress Test — Concurrent Request Behavior

**Why concurrency matters:** A single retail store might have 8 cameras. All 8 could send an image simultaneously when a shelf restock event is triggered. If the server is single-threaded (default Flask dev server), it queues requests and latency jumps 8×.

**`asyncio` + `httpx`:** We fire `CONCURRENT_REQUESTS` requests at the same instant using `asyncio.gather()` and measure P50/P95/P99 latency. The production API uses Gunicorn with 4 workers — this test reveals whether 4 workers is enough.

**Expected behavior for a well-configured server:**
- 1 concurrent request: latency ≈ single-request baseline
- 4 concurrent requests: latency ≈ 1.1–1.3× (fits within 4 workers)
- 8 concurrent requests: latency ≈ 1.8–2.5× (queue buildup)
- 20 concurrent requests: latency ≈ 5–10× (significant queuing)

In [ ]:
# ── Async stress test: 20 concurrent requests ─────────────────────────────────
# NOTE: asyncio.run() / nest_asyncio needed in Jupyter (event loop already running)
CONCURRENT_REQUESTS = 10 if QUICK_MODE else 20

try:
    import nest_asyncio
    nest_asyncio.apply()  # allows asyncio.run inside Jupyter's existing event loop
    HAS_NEST_ASYNCIO = True
except ImportError:
    HAS_NEST_ASYNCIO = False
    print('nest_asyncio not installed. Run: pip install nest_asyncio')
    print('The async stress test cell will be skipped.')

async_results = []

if HAS_HTTPX and (HAS_NEST_ASYNCIO or True):  # True = try anyway, may work depending on kernel
    # Use the mock requests for QUICK_MODE (httpx is async-native)
    async def send_detect_request(client, image_bytes, request_id):
        """Single async /detect request; returns (request_id, latency_ms, n_detections)."""
        if QUICK_MODE:
            # Simulate the mock response with asyncio sleep
            await asyncio.sleep(random.uniform(0.010, 0.045))
            return request_id, random.uniform(15, 45), random.randint(1, 6)
        t0 = time.perf_counter()
        response = await client.post(
            f'{SERVICE_URL}/detect',
            content=image_bytes,
            headers={'Content-Type': 'image/jpeg'},
            timeout=TIMEOUT_S,
        )
        latency_ms = (time.perf_counter() - t0) * 1000
        n_det = len(response.json().get('detections', [])) if response.status_code == 200 else 0
        return request_id, latency_ms, n_det

    async def run_concurrent_batch(n_concurrent):
        """Fire n_concurrent requests simultaneously and collect latencies."""
        images = [make_test_image_bytes(seed=i)[0] for i in range(n_concurrent)]
        async with httpx.AsyncClient() as client:
            tasks = [
                send_detect_request(client, img, req_id)
                for req_id, img in enumerate(images)
            ]
            # asyncio.gather fires all tasks concurrently
            results = await asyncio.gather(*tasks, return_exceptions=True)
        return [r for r in results if not isinstance(r, Exception)]

    loop = asyncio.get_event_loop()
    raw_results = loop.run_until_complete(run_concurrent_batch(CONCURRENT_REQUESTS))
    async_results = [(req_id, lat_ms, n_det) for req_id, lat_ms, n_det in raw_results]
    latencies = [r[1] for r in async_results]

    print(f'Async stress test ({CONCURRENT_REQUESTS} simultaneous requests)')
    print(f'  Completed:      {len(async_results)}/{CONCURRENT_REQUESTS}')
    print(f'  P50 latency:    {np.percentile(latencies, 50):.1f}ms')
    print(f'  P95 latency:    {np.percentile(latencies, 95):.1f}ms')
    print(f'  P99 latency:    {np.percentile(latencies, 99):.1f}ms')
    print(f'  Max latency:    {max(latencies):.1f}ms')
else:
    print('[Async stress test skipped — install httpx and nest_asyncio]')

## 5. Latency vs Concurrency Chart

**The scaling curve** shows how median latency grows as we increase the number of simultaneous requests. The inflection point (where latency starts growing superlinearly) reveals the maximum concurrency the server handles gracefully.

**For the ProductionCV deployment decision:**
- If P95 latency stays < 50ms up to N concurrent requests → N is the safe operating limit
- Scale horizontally (more server replicas) once P95 exceeds 50ms at expected peak load

In [ ]:
# ── Latency vs concurrency sweep ──────────────────────────────────────────────
# Test concurrency levels 1, 5, 10, 20 (reduced for QUICK_MODE)
# Each level fires REPS_PER_LEVEL batches and averages the P50 latency.
CONCURRENCY_LEVELS = [1, 3, 5, 10] if QUICK_MODE else [1, 5, 10, 20]
REPS_PER_LEVEL     = 3 if QUICK_MODE else 5

if not (HAS_HTTPX):
    print('[Skipped — install httpx to run this cell]')
else:
    sweep_results = {level: {'p50': [], 'p95': []} for level in CONCURRENCY_LEVELS}

    for level in CONCURRENCY_LEVELS:
        for rep in range(REPS_PER_LEVEL):
            if QUICK_MODE:
                # Synthetic latency: base + queuing delay
                base = 20
                queue_factor = max(1.0, level / 4)  # 4 hypothetical workers
                lats = [base * queue_factor + random.uniform(-3, 8) for _ in range(level)]
            else:
                loop = asyncio.get_event_loop()
                raw = loop.run_until_complete(run_concurrent_batch(level))
                lats = [r[1] for r in raw]
            sweep_results[level]['p50'].append(np.percentile(lats, 50))
            sweep_results[level]['p95'].append(np.percentile(lats, 95))

    # Aggregate across reps
    levels      = CONCURRENCY_LEVELS
    median_p50  = [np.mean(sweep_results[l]['p50']) for l in levels]
    median_p95  = [np.mean(sweep_results[l]['p95']) for l in levels]

    # Print table
    print('Concurrency | P50 latency (ms) | P95 latency (ms)')
    print('-' * 48)
    for l, p50, p95 in zip(levels, median_p50, median_p95):
        flag = '⚠' if p95 > 50 else '✓'
        print(f'{l:>11}  |  {p50:>14.1f}  |  {p95:>14.1f}  {flag}')

    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(levels, median_p50, 'o-', color='steelblue', linewidth=2, label='P50 (median)')
    ax.plot(levels, median_p95, 's-', color='coral',     linewidth=2, label='P95')
    ax.axhline(50, color='red', linestyle='--', alpha=0.7, label='50ms SLA target')
    ax.fill_between(levels, median_p50, median_p95, alpha=0.1, color='coral')

    # Mark where P95 crosses the SLA
    for l, p95 in zip(levels, median_p95):
        if p95 > 50:
            ax.annotate(f'SLA breach\nat {l} concurrent',
                        xy=(l, p95), xytext=(l + 0.5, p95 + 5),
                        arrowprops=dict(arrowstyle='->', color='red'),
                        color='red', fontsize=8)
            break

    ax.set_xlabel('Concurrent Requests')
    ax.set_ylabel('Latency (ms)')
    ax.set_title('API Latency vs Concurrency\n(inflection = server capacity limit)')
    ax.set_xticks(levels)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()